# Pipeline único — de los datos crudos al submit

Un solo notebook para todo: preprocesamiento, feature engineering, normalización,
búsqueda de hiperparámetros con Optuna, entrenamiento final y subida a Kaggle.

## Reanudación tras la muerte de una spot

Cada etapa cara escribe su resultado al bucket y **se saltea sola si ya está hecha**.
Cuando Google te mate la máquina, levantás una nueva y corrés
**Kernel → Restart Kernel and Run All Cells**: el notebook detecta lo ya calculado,
lo carga, y sigue desde donde se cortó.

Dos protecciones que hacen que eso sea confiable:

- **Escritura atómica.** Todo se escribe a un `.tmp` y recién al terminar se renombra.
  Un corte a mitad de camino no deja nunca un archivo con el nombre definitivo.
- **Validación al leer.** Un parquet válido termina con los bytes `PAR1`. Si un
  checkpoint está truncado se detecta y se recalcula, en vez de explotar tres
  etapas más adelante.

El study de Optuna vive en SQLite dentro del bucket y se respalda cada N trials,
así que también retoma donde quedó.

## Todo se controla desde la celda de palancas

Los nombres de archivo arrastran la configuración, así que **cambiar una palanca
crea un experimento nuevo sin pisar el anterior**, y volver a la configuración
anterior reusa sus checkpoints.

## 0 — Ambiente

In [ ]:
# Dependencias. uv es mucho mas rapido que pip para esto.
!pip install -q uv
!uv pip install -q pyarrow polars lightgbm pandas optuna kaggle scikit-learn

In [ ]:
import sys
from pathlib import Path

# El modulo labo3 vive al lado de este notebook, dentro del repo.
_aqui = Path.cwd()
for _c in (_aqui, *_aqui.parents):
    if (_c / "src" / "pipe_unico" / "labo3.py").exists():
        sys.path.insert(0, str(_c / "src" / "pipe_unico")); break
    if (_c / "labo3.py").exists():
        sys.path.insert(0, str(_c)); break
else:
    raise RuntimeError("No encuentro labo3.py. Corre este notebook desde el repo clonado.")

import labo3 as L

L.descargar_datasets()      # baja los crudos que falten (atomico)
L.instalar_kaggle()         # deja ~/.kaggle/kaggle.json con permisos 600
print(f"\nBucket: {L.BUCKET}")
print(f"Archivos .tmp de corridas interrumpidas eliminados: {L.limpiar_tmp()}")

## 1 — Palancas

Todo lo configurable está acá. Cada palanca entra en el nombre de los archivos,
así que dos configuraciones distintas nunca se pisan.

In [ ]:
# ── Celda de parametros (papermill) ────────────────────────────────────────
# El runner de experimentos inyecta aca un OVERRIDES con las palancas a cambiar.
# Corriendo el notebook a mano queda vacio y valen los defaults de la celda de abajo.
OVERRIDES = {}


In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    """Periodos AAAAMM consecutivos: rango_meses(201701, 201703) -> [201701, 201702, 201703]."""
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    # ══ ETAPA 1 — PREPROCESAMIENTO ═════════════════════════════════════════
    # 'A' = una fila por cliente-producto-mes | 'B' = una fila por producto-mes.
    # 'A' da mucha mas senial (el patron de cada cliente) a costa de 20x mas filas.
    'agrupamiento':   'A',

    # Meses sin venta DENTRO de la vida del producto: 'zero' los pone en 0,
    # 'null' los deja nulos. 0 dice "no vendio"; nulo dice "no sabemos".
    'completado':     'zero',

    # 'lifecycle' = cada producto vive entre su primera y ultima venta.
    # 'full' = malla completa para todos, incluso antes de existir.
    'densificacion':  'lifecycle',

    # True = solo los 780 productos que hay que predecir. Achica mucho el dataset,
    # pero el modelo pierde la senial de los productos que no se entregan.
    'solo_target':    False,

    # ══ ETAPA 2 — FEATURE ENGINEERING ══════════════════════════════════════
    'max_lags':       24,        # tn0..tn24 por fila
    'metodo_norm':    'recta',   # 'recta' | 'zscore' | 'minmax' | 'media'
    'salto_deltas':   2,         # tn{k}_delta = tn{k}_norm - tn{k+salto}_norm
    'n_periodos':     42,        # cuantos meses del panel generar
    'horizonte':      2,         # la clase es tn(t + horizonte)

    # ══ ETAPA 3 — DATOS PARA EL MODELO ═════════════════════════════════════
    # 'clase_tn'       -> predecir toneladas directo
    # 'clase_tn_norm'  -> predecir la serie normalizada (quita nivel y tendencia)
    # 'clase_tn_delta' -> predecir el CAMBIO respecto de hoy
    'target':         'clase_tn_norm',

    'meses_train':    rango_meses(201701, 201905),
    'meses_val':      [201907, 201908],
    'meses_test':     [201910],
    'reentrenar_con_val_para_test': True,

    # Poblacion de clientes para ENTRENAR. Val y test usan SIEMPRE todos, para que
    # la metrica sea comparable entre experimentos.
    #   'todos'    -> sin filtro
    #   'muestreo' -> 1 de cada N clientes, por hash (subconjunto REPRESENTATIVO)
    #   'top'      -> los N clientes de mayor tn (CAMBIA LA POBLACION, ver nota abajo)
    'filtro_clientes': 'muestreo',
    'clientes_n':      4,

    # ══ ETAPA 4 — OPTUNA ═══════════════════════════════════════════════════
    # OJO: el TPESampler sortea al azar los primeros 10 trials y recien despues
    # modela. Con n_trials <= 10 la busqueda bayesiana NO se activa.
    'n_trials':          40,
    'trials_arranque':   10,
    'techo_arboles':     500,
    'objective_lgbm':    'regression',   # 'regression'|'regression_l1'|'tweedie'|'poisson'
    'regularizacion':    'normal',       # 'normal' | 'fuerte'
    'semilla':           102191,
    'backup_cada_n_trials': 5,
    'wape_por_producto': True,           # asi evalua la competencia

    # Peso por recencia: los meses recientes pesan mas en el entrenamiento.
    # 1.0 = todos los meses igual. 0.97 = cada mes hacia atras pesa 3% menos.
    'decay_recencia':    1.0,

    # Submuestreo de FILAS al azar (0-1). None = todas. Rompe las series, asi que
    # es preferible usar filtro_clientes, que conserva la historia completa.
    'sampling_frac':     None,

    # Si objective_lgbm='tweedie', buscar tambien su parametro de varianza.
    'tweedie_optimizar': False,

    # ══ ETAPA 5 — ENTREGA ══════════════════════════════════════════════════
    'periodo_objetivo':   202002,
    'semillas_ensemble':  [102191],
    'clip_min':           0.0,

    # Kaggle exige EXACTAMENTE las filas de product_id_apredecir201912.txt. Los
    # productos de esa lista que no lleguen a tener prediccion (murieron antes del
    # periodo de inferencia) se rellenan asi:
    #   'cero'  -> 0 toneladas. Correcto si el producto dejo de venderse de verdad.
    #   'naive' -> repetir su ultimo tn conocido. Mejor si solo falto el dato.
    'relleno_faltantes':  'naive',
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit':             False,     # True = sube solo al terminar
    'mensaje_submit':     None,

    # ══ CONTROL DE CHECKPOINTS ═════════════════════════════════════════════
    # Etapas a recalcular aunque ya existan: 'prep', 'fe', 'final', 'submission'.
    # Vacio = reusar todo lo que haya (lo normal).
    'forzar':         set(),
}

# Los overrides del runner pisan lo de arriba. Asi un mismo notebook sirve para
# todos los experimentos de la cola, cambiando solo las palancas que interesan.
if OVERRIDES:
    _desconocidas = set(OVERRIDES) - set(PARAM)
    if _desconocidas:
        raise ValueError(f"Palancas inexistentes en OVERRIDES: {sorted(_desconocidas)}")
    PARAM.update(OVERRIDES)
    print(f"OVERRIDES aplicados: {OVERRIDES}")

print(f"agrupamiento : {PARAM['agrupamiento']}   target: {PARAM['target']}")
print(f"clientes     : {PARAM['filtro_clientes']} ({PARAM['clientes_n']})")
print(f"trials       : {PARAM['n_trials']}   submit: {PARAM['submit']}")

## 2 — Preprocesamiento y feature engineering

Las dos etapas caras. Si los parquet ya existen para esta configuración, se saltean:
verás `ya existe, se saltea`. Si alguno quedó truncado por una spot muerta, se detecta
y se recalcula.

In [ ]:
cfg_prep = L.PreprocessingConfig(
    group_mode                  = PARAM['agrupamiento'],
    missing_strategy            = PARAM['completado'],
    densify_strategy            = PARAM['densificacion'],
    filter_target_products_only = PARAM['solo_target'],
)

path_prep = L.etapa_preprocesar(cfg_prep, forzar='prep' in PARAM['forzar'])

path_fe = L.etapa_fe(
    path_prep,
    max_lags      = PARAM['max_lags'],
    metodo        = PARAM['metodo_norm'],
    salto         = PARAM['salto_deltas'],
    n_periodos    = PARAM['n_periodos'],
    horizonte     = PARAM['horizonte'],
    forzar        = 'fe' in PARAM['forzar'],
)

GRANULARIDAD = L.granularidad_de(path_prep.name)
print(f"\nDataset  : {path_fe.name}")
print(f"Granularidad: {GRANULARIDAD}   ({'producto-cliente' if GRANULARIDAD=='pc' else 'solo producto'})")

## 3 — Nombre del experimento

In [ ]:
import json, os, re, shutil
import numpy as np
import polars as pl

TARGETS_VALIDOS = {'clase_tn': 'nivel', 'clase_tn_norm': 'norm', 'clase_tn_delta': 'delta'}
TARGET      = PARAM['target']
if TARGET not in TARGETS_VALIDOS:
    raise ValueError(f"target invalido: {TARGET!r}. Opciones: {list(TARGETS_VALIDOS)}")
TARGET_KIND = TARGETS_VALIDOS[TARGET]
METODO      = PARAM['metodo_norm']
H           = PARAM['horizonte']

# El nombre arrastra TODO lo que cambia los datos o el espacio de busqueda. Dos
# corridas con el mismo nombre son la misma; con nombres distintos no se pisan.
_tag_cli = {'todos': '', 'muestreo': f"__cli1de{PARAM['clientes_n']}",
            'top': f"__cliTop{PARAM['clientes_n']}"}[PARAM['filtro_clientes']]
_v, _t = PARAM['meses_val'], PARAM['meses_test']

EXPERIMENTO = (f"{path_fe.name[len('preprocesado_'):-len('.parquet')]}"
               f"__y-{TARGET_KIND}__{PARAM['objective_lgbm']}"
               f"__val{_v[0]}-{_v[-1]}_test{_t[0]}-{_t[-1]}"
               f"{_tag_cli}__arb{PARAM['techo_arboles']}")

DIR_OUT   = L.RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)
DB_LOCAL  = Path.home() / f"optuna_{EXPERIMENTO}.db"
DB_BUCKET = L.DIR_DB / f"{EXPERIMENTO}.db"

# Reanudacion del study: si la VM es nueva, el .db local no existe pero el del
# bucket si. Traerlo es lo que hace que no se pierdan los trials ya corridos.
if DB_BUCKET.exists() and not DB_LOCAL.exists():
    shutil.copy(DB_BUCKET, DB_LOCAL)
    print(f"Study recuperado del bucket: {DB_BUCKET.name}")
STORAGE = f"sqlite:///{DB_LOCAL}"

print(f"EXPERIMENTO: {EXPERIMENTO}")
print(f"Salidas    : {DIR_OUT}")
print("\nEstado de los checkpoints:")
for _n, _p in [('preprocesado', path_prep), ('feature engineering', path_fe),
               ('study optuna', DB_BUCKET), ('resultado.json', DIR_OUT / 'resultado.json'),
               ('predicciones', DIR_OUT / 'pred_infer.parquet'),
               ('submission', DIR_OUT / f"submission_{PARAM['periodo_objetivo']}.csv")]:
    print(f"   {'OK  ' if _p.exists() else '--  '} {_n:22} {_p.name}")

## 4 — Carga, split y población de clientes

In [ ]:
# Float32 para las features (LightGBM las bineriza igual) y Float64 para las columnas
# que entran en la aritmetica de toneladas: ahi el round-trip necesita la precision.
CTX_F64 = {'B0', 'B1', 'tn0_norm', 'tn0', 'clase_tn', 'clase_tn_norm', 'clase_tn_delta'}

lf = pl.scan_parquet(path_fe)
_schema  = lf.collect_schema()
COLS_ALL = list(_schema.keys())
if TARGET not in COLS_ALL:
    raise ValueError(f"El dataset no tiene {TARGET!r}. Hay: "
                     f"{[c for c in COLS_ALL if c.startswith('clase_')]}")
_a_f32 = [c for c, t in _schema.items() if t == pl.Float64 and c not in CTX_F64]
lf = lf.with_columns([pl.col(c).cast(pl.Float32) for c in _a_f32])

periodos    = sorted(lf.select('periodo').unique().collect()['periodo'].to_list())
MESES_INFER = periodos[-H:]
_es_eval  = pl.col('periodo').is_in(sorted(set(PARAM['meses_val']) | set(PARAM['meses_test'])))
_es_train = pl.col('periodo').is_in(sorted(set(PARAM['meses_train'])))

# ── Poblacion de clientes para train ────────────────────────────────────────
# El filtro se aplica SOLO a train. Val y test conservan todos los clientes: si no,
# el WAPE dejaria de ser comparable entre experimentos.
_modo_cli = PARAM['filtro_clientes']
_hay_cli  = 'customer_id' in COLS_ALL
_cli_ok   = None

if _modo_cli != 'todos' and not _hay_cli:
    print(f"[aviso] granularidad '{GRANULARIDAD}': no hay customer_id, el filtro no aplica")
elif _modo_cli == 'muestreo':
    # hash deterministico: el mismo cliente cae siempre del mismo lado
    _cli_ok = pl.col('customer_id').hash(seed=PARAM['semilla']) % int(PARAM['clientes_n']) == 0
    print(f"Clientes en train: 1 de cada {PARAM['clientes_n']} (muestra representativa)")
elif _modo_cli == 'top':
    _col_tn = next((c for c in ('tn0', 'tn') if c in COLS_ALL), None)
    if _col_tn is None:
        raise RuntimeError("No hay columna de toneladas para rankear clientes.")
    # El ranking sale SOLO de train: elegir mirando val/test seria filtrar el futuro.
    _rk = (lf.filter(_es_train).group_by('customer_id')
             .agg(pl.col(_col_tn).sum().alias('_tn'))
             .sort('_tn', descending=True).head(int(PARAM['clientes_n'])).collect())
    _cli_ok = pl.col('customer_id').is_in(_rk['customer_id'].to_list())
    _tot = float(lf.filter(_es_train).select(pl.col(_col_tn).sum()).collect().item())
    print(f"Clientes en train: top {_rk.height} por {_col_tn} "
          f"({100*float(_rk['_tn'].sum())/_tot:.1f}% del volumen)")
    print("   OJO: esto CAMBIA LA POBLACION. El modelo entrena solo con clientes")
    print("   grandes pero se lo evalua contra todos, incluidos los chicos que no vio.")

_filtro = pl.col(TARGET).is_not_null()
if _cli_ok is not None:
    _filtro = _filtro & (_es_eval | (_es_train & _cli_ok))

df_infer = lf.filter(pl.col(TARGET).is_null() & pl.col('periodo').is_in(MESES_INFER)).collect()
df_sup   = lf.filter(_filtro).collect()

_set_sup    = set(df_sup['periodo'].unique().to_list())
MESES_TRAIN = sorted(set(PARAM['meses_train']) & _set_sup)
MESES_VAL   = sorted(set(PARAM['meses_val'])   & _set_sup)
MESES_TEST  = sorted(set(PARAM['meses_test'])  & _set_sup)

print(f"\nSupervisado: {df_sup.height:,} filas   inferencia: {df_infer.height:,} filas")
print(f"train {len(MESES_TRAIN)} meses | val {MESES_VAL} | test {MESES_TEST} | infer {MESES_INFER}")

# ── Control de leakage ──────────────────────────────────────────────────────
# Entre el ultimo mes de train y el primero de val tiene que haber al menos
# `horizonte` meses de separacion: si no, la clase de train ya vio el periodo de val.
def _idx(p): return (p // 100) * 12 + p % 100
if MESES_VAL and _idx(max(MESES_TRAIN)) + H > _idx(min(MESES_VAL)):
    raise RuntimeError(f"LEAKAGE: max(train)={max(MESES_TRAIN)} + horizonte {H} "
                       f"invade val={min(MESES_VAL)}")
if MESES_TEST and MESES_VAL and _idx(max(MESES_VAL)) + H > _idx(min(MESES_TEST)):
    raise RuntimeError(f"LEAKAGE: max(val)={max(MESES_VAL)} + horizonte {H} "
                       f"invade test={min(MESES_TEST)}")
if set(MESES_INFER) & (set(MESES_TRAIN) | set(MESES_VAL) | set(MESES_TEST)):
    raise RuntimeError("LEAKAGE: los meses de inferencia se solapan con train/val/test")
print("Control de leakage: OK")

In [ ]:
import gc

# ── Que columnas son features ───────────────────────────────────────────────
# Se excluye todo lo que sea identificador, la clase en cualquiera de sus formas, y
# los parametros de normalizacion (B0/B1 reconstruyen la clase: son leakage directo).
NO_FEATURE = ({'periodo', 'product_id', 'customer_id', 'Agrupacion_ID',
               'B0', 'B1', 'p_B0', 'p_B1', 'first_sell_in_period'}
              | {c for c in COLS_ALL if c.startswith('clase_')})
FEATURES     = [c for c in COLS_ALL if c not in NO_FEATURE]
CAT_FEATURES = [c for c in ('cat1', 'cat2', 'cat3', 'brand') if c in FEATURES]

IDS      = [c for c in ('product_id', 'customer_id', 'Agrupacion_ID') if c in COLS_ALL]
# tn0 (lo vendido en el mes actual) viaja para poder rellenar con el naive los
# productos de la lista oficial de Kaggle que no lleguen a tener prediccion.
COLS_CTX = [c for c in ('B0', 'B1', 'tn0_norm', 'tn0', 'clase_tn') + tuple(IDS)
            if c in COLS_ALL]
_cols    = sorted(set(FEATURES + COLS_CTX + [TARGET, 'periodo']))
_cats    = [c for c in CAT_FEATURES if c in _cols]

# Las categoricas se castean en POLARS antes de pasar a pandas: asi llegan como
# 'category' y no como object dtype (un str de Python por celda, GB de basura).
df_pd = (df_sup.select(_cols)
               .with_columns([pl.col(c).cast(pl.Categorical) for c in _cats])
               .to_pandas())
df_infer_pd = (df_infer.select([c for c in _cols if c in df_infer.columns])
                       .with_columns([pl.col(c).cast(pl.Categorical)
                                      for c in _cats if c in df_infer.columns])
                       .to_pandas())
for c in CAT_FEATURES:
    df_pd[c] = df_pd[c].astype('category')
    if c in df_infer_pd.columns:
        df_infer_pd[c] = (df_infer_pd[c].astype('category')
                          .cat.set_categories(df_pd[c].cat.categories))

print(f"{len(FEATURES)} features   categoricas: {CAT_FEATURES}")
print(f"df_pd: {df_pd.shape} ({df_pd.memory_usage(deep=True).sum()/1e9:.2f} GB)   "
      f"inferencia: {df_infer_pd.shape}")

# polars ya no se usa: liberarlo evita tener el dataset duplicado en RAM justo
# cuando LightGBM necesita el espacio.
del df_sup, df_infer
gc.collect()

In [ ]:
# Utilidades de periodo (enteros AAAAMM), de 03_Optuna.

def a_indice_mes(p: int) -> int:
    return (p // 100) * 12 + (p % 100) - 1

def desplazar_meses(p: int, k: int) -> int:
    m = a_indice_mes(p) + k
    return (m // 12) * 100 + (m % 12) + 1

## 5 — Métrica: WAPE en toneladas

In [ ]:
def reconstruir_nivel(pred, df_ctx: pl.DataFrame) -> np.ndarray:
    """Pasa la prediccion del modelo a toneladas, segun la variable respuesta elegida.

    df_ctx debe traer B0, B1 y (si target='clase_tn_delta') tn0_norm, alineadas por fila.
    """
    pred = np.asarray(pred, dtype=np.float64)

    if TARGET_KIND == 'nivel':
        return pred

    if TARGET_KIND == 'delta':
        # clase_tn_delta = clase_tn_norm - tn0_norm  ->  volvemos a la escala normalizada
        pred = pred + df_ctx['tn0_norm'].to_numpy().astype(np.float64)

    B0 = df_ctx['B0'].to_numpy().astype(np.float64)
    B1 = df_ctx['B1'].to_numpy().astype(np.float64)

    if METODO == 'recta':
        # inverso de: norm = valor - (B0 + B1 * lag);  la clase esta en lag = -2
        return pred + (B0 + B1 * (-2.0))
    # zscore | minmax | media: inverso de norm = (valor - B0) / B1, con el mismo B1 seguro
    B1_safe = np.where((B1 == 0) | ~np.isfinite(B1), 1.0, B1)
    return pred * B1_safe + B0


def wape(y_real_tn, y_pred_tn, product_ids=None, por_producto=True) -> float:
    """WAPE en toneladas. Con por_producto=True agrega por product_id primero
    (asi lo evalua la competencia: el error es sobre el total vendido de cada producto)."""
    y_real = np.asarray(y_real_tn, dtype=np.float64)
    y_pred = np.maximum(np.asarray(y_pred_tn, dtype=np.float64), 0.0)  # no hay ventas negativas

    if por_producto and product_ids is not None:
        ids = np.asarray(product_ids)
        orden, inv = np.unique(ids, return_inverse=True)
        y_real = np.bincount(inv, weights=y_real, minlength=len(orden))
        y_pred = np.bincount(inv, weights=y_pred, minlength=len(orden))

    den = np.abs(y_real).sum()
    return float('nan') if den == 0 else float(np.abs(y_real - y_pred).sum() / den)

## 6 — Espacio de búsqueda y funciones de entrenamiento

Los hiperparámetros que querés **fijos** van como valores literales; los que querés
**buscar**, por `trial.suggest_*`. Un `suggest` con rango de un solo punto no fija
nada: le declara a Optuna una dimensión inútil y queda registrado en la base con su
tipo, lo que después impide cambiarlo sin borrar el study.

In [ ]:
# Funciones de entrenamiento y evaluacion, tomadas de 03_Optuna sin cambios.
# La preparacion de df_pd / df_infer_pd ya la hizo la celda anterior.
import lightgbm as lgb

def pesos_recencia(periodos_serie, decay):
    """Peso por recencia: el mes mas nuevo pesa 1, cada mes hacia atras decae x`decay`."""
    if decay is None:
        return None
    ps = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(ps)}
    n = len(ps)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values


def espacio_hiper(trial):
    base = {
        'objective':      PARAM['objective_lgbm'],
        'metric':         'mae',
        'verbosity':      -1,
        'boosting_type':  'gbdt',
        'seed':           PARAM['semilla'],
        'subsample_freq': 1,
        'n_jobs':         -1,
    }
    # Techo de arboles: el tiempo de LightGBM crece casi lineal en n_estimators, asi
    # que este numero es la palanca mas directa sobre la duracion de la busqueda.
    # min(100, ...) para que un techo bajo (pruebas rapidas) no rompa el suggest_int:
    # Optuna exige low <= high y el piso natural del rango es 100 arboles.
    _techo = int(PARAM.get('techo_arboles', 2000))
    _hi_f = min(800, _techo)
    if PARAM['regularizacion'] == 'fuerte':
        base.update({
            'num_leaves':        trial.suggest_int('num_leaves', 8, 64),
            'max_depth':         trial.suggest_int('max_depth', 3, 7),
            'learning_rate':     trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'n_estimators':      trial.suggest_int('n_estimators', min(100, _hi_f), _hi_f),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
            'subsample':         trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha':         trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        })
    else:
        base.update({
            'num_leaves':        trial.suggest_int('num_leaves', 20, 300),
            'max_depth':         trial.suggest_int('max_depth', 3, 12),
            'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'n_estimators':      trial.suggest_int('n_estimators', min(100, _techo), _techo),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        })
    if PARAM['objective_lgbm'] == 'tweedie' and PARAM['tweedie_optimizar']:
        base['tweedie_variance_power'] = trial.suggest_float('tweedie_variance_power', 1.1, 1.9)
    return base


def entrenar(params, meses_tr):
    """Entrena con las filas cuyo periodo esta en `meses_tr`. Devuelve el modelo.

    Materializa UNA sola copia, y ya muestreada: primero se calcula el indice de filas
    (barato, es un array de posiciones) y recien despues se extraen las columnas.
    Al reves -- cortar por mes y despues samplear, como estaba antes -- se aloca el
    slice completo y despues se tira la mayor parte; con 8,6M filas eran 21 GB de mas,
    y sumados a df_pd + la matriz interna de LightGBM el kernel moria por OOM.
    """
    idx = df_pd.index[df_pd['periodo'].isin(meses_tr)]
    if len(idx) == 0:
        raise ValueError(f'Sin filas de entrenamiento para los meses {meses_tr}')

    frac = PARAM.get('sampling_frac')
    if frac is not None and frac < 1.0:
        rng = np.random.default_rng(PARAM['semilla'])
        idx = idx[rng.choice(len(idx), size=int(len(idx) * frac), replace=False)]

    X = df_pd.loc[idx, FEATURES]
    y = df_pd.loc[idx, TARGET]
    w = pesos_recencia(df_pd.loc[idx, 'periodo'], PARAM['decay_recencia'])

    modelo = lgb.LGBMRegressor(**params)
    modelo.fit(X, y, sample_weight=w, categorical_feature=CAT_FEATURES)
    del X, y, w
    gc.collect()
    return modelo


def predecir(modelo, df_eval):
    """Devuelve el DataFrame de evaluacion con ids, prediccion cruda y toneladas."""
    pred = modelo.predict(df_eval[FEATURES])
    ctx = pl.from_pandas(df_eval[[c for c in COLS_CTX if c in df_eval.columns]]
                         .reset_index(drop=True))
    pred_tn = np.maximum(reconstruir_nivel(pred, ctx), 0.0)  # no hay ventas negativas

    out = df_eval[IDS + ['periodo']].copy()
    # A que mes corresponde la prediccion: la fila es de t, se predice t+horizonte.
    out['periodo_objetivo'] = out['periodo'].map(lambda p: desplazar_meses(p, H))
    out['y_pred_target'] = pred
    out['tn_pred'] = pred_tn
    if 'clase_tn' in df_eval.columns:
        out['tn_real'] = df_eval['clase_tn'].values
        out['y_real_target'] = df_eval[TARGET].values
    return out


def evaluar(modelo, meses_ev, por_mes=True):
    """WAPE en toneladas sobre `meses_ev`. Devuelve (wape_global, {mes: wape}, df_pred).

    OJO: los meses de val/test NUNCA se muestrean en la carga, asi que este WAPE se
    mide sobre TODAS las filas de esos meses y es comparable entre experimentos.
    """
    df_ev = df_pd[df_pd['periodo'].isin(meses_ev)]
    if len(df_ev) == 0:
        return float('nan'), {}, None
    pred = predecir(modelo, df_ev)

    por_mes_d = {}
    if por_mes:
        for m in sorted(meses_ev):
            sub = pred[pred['periodo'] == m]
            if len(sub):
                por_mes_d[int(m)] = wape(sub['tn_real'], sub['tn_pred'],
                                         sub['product_id'], PARAM['wape_por_producto'])
    # WAPE global: todos los meses de evaluacion juntos (asi lo mide la competencia).
    glob = wape(pred['tn_real'], pred['tn_pred'], pred['product_id'],
                PARAM['wape_por_producto'])
    return glob, por_mes_d, pred


def objective(trial):
    """Optuna minimiza el WAPE en TONELADAS sobre MESES_VAL. Test no se toca aca."""
    modelo = entrenar(espacio_hiper(trial), MESES_TRAIN)
    score, por_mes, _ = evaluar(modelo, MESES_VAL)
    if np.isnan(score):
        raise optuna.TrialPruned()
    for m, v in por_mes.items():
        trial.set_user_attr(f'wape_val_{m}', v)
    return float(score)

## 7 — Búsqueda bayesiana

El study vive en SQLite dentro del bucket y se respalda cada `backup_cada_n_trials`.
Si Google mata la máquina en el trial 32, la próxima corrida arranca desde el último
respaldo en vez de desde cero.

In [ ]:
import optuna
from tqdm.auto import tqdm
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=PARAM['semilla'],
                                       n_startup_trials=PARAM['trials_arranque']),
    study_name=EXPERIMENTO,
    storage=STORAGE,
    load_if_exists=True,
)
_previos = len(study.trials)
_faltan  = max(0, PARAM['n_trials'] - _previos)
print(f"Trials ya hechos: {_previos}   faltan: {_faltan}   (objetivo: {PARAM['n_trials']})")

def respaldar_study():
    """Copia el .db al bucket. Atomico: una interrupcion no deja un backup corrupto."""
    try:
        tmp = DB_BUCKET.with_suffix('.db.tmp')
        shutil.copy(DB_LOCAL, tmp)
        tmp.replace(DB_BUCKET)
        return True
    except Exception as e:
        print(f"   [aviso] no se pudo respaldar: {e}")
        return False

if _faltan == 0:
    print("Nada que buscar: el study ya llego al objetivo.")
else:
    _cada = max(1, int(PARAM['backup_cada_n_trials']))
    with tqdm(total=_faltan, desc="Optuna") as _barra:
        def _cb(st, tr):
            _barra.update(1)
            if len(st.trials) % _cada == 0:
                respaldar_study()
        try:
            study.optimize(objective, n_trials=_faltan, callbacks=[_cb])
        except KeyboardInterrupt:
            print("\nInterrumpido a mano: se respalda lo hecho.")
        finally:
            if respaldar_study():
                print(f"Study respaldado en {DB_BUCKET}")

_ok = [t for t in study.trials if t.value is not None]
print(f"\nTrials totales: {len(study.trials)}   completos: {len(_ok)}")
if not _ok:
    raise RuntimeError("Ningun trial completo. Revisa el espacio de busqueda.")
print(f"Mejor WAPE val: {study.best_value:.5f}")
for k, v in study.best_params.items():
    print(f"   {k:22} {v}")

## 8 — Modelo final y evaluación en test

Checkpoint: si `resultado.json` ya existe para este experimento, se carga en lugar de
reentrenar.

In [ ]:
import pandas as pd
import lightgbm as lgb

_path_res = DIR_OUT / 'resultado.json'

def _entrenar_y_evaluar():
    mejores = espacio_hiper(optuna.trial.FixedTrial(study.best_params))

    modelo_val = entrenar(mejores, MESES_TRAIN)
    wape_val, wape_val_mes, _ = evaluar(modelo_val, MESES_VAL)
    print(f"VALIDATION {MESES_VAL}: WAPE {wape_val:.5f}")

    meses_fit = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
    modelo_test = entrenar(mejores, meses_fit)
    wape_test, wape_test_mes, _ = evaluar(modelo_test, MESES_TEST)
    print(f"TEST (holdout) {MESES_TEST}: WAPE {wape_test:.5f}")
    _brecha = wape_test - wape_val
    print(f"Brecha test - val: {_brecha:+.5f}"
          + ("   <- el test empeora bastante: posible sobreajuste a val" if _brecha > 0.02 else ""))

    # Baseline naive: repetir lo vendido hoy. Es la vara minima; si el modelo no le
    # gana, todo el pipeline no esta aportando nada.
    def _naive(meses):
        _n = df_pd[df_pd['periodo'].isin(meses)]
        if len(_n) == 0 or 'tn0' not in _n.columns:
            return float('nan')
        return wape(_n['clase_tn'].to_numpy(), _n['tn0'].to_numpy(),
                    _n['product_id'].to_numpy(), PARAM['wape_por_producto'])
    naive_val, naive_test = _naive(MESES_VAL), _naive(MESES_TEST)
    print(f"\nBaseline naive:  val {naive_val:.5f}   test {naive_test:.5f}")
    print(f"Mejora vs naive: val {100*(naive_val-wape_val)/naive_val:+.1f}%"
          f"   test {100*(naive_test-wape_test)/naive_test:+.1f}%")

    L.escribir_json({
        'experimento': EXPERIMENTO, 'dataset_fe': path_fe.name,
        'target': TARGET, 'target_kind': TARGET_KIND, 'metodo_normalizacion': METODO,
        'horizonte': H, 'objective_lgbm': PARAM['objective_lgbm'],
        'agrupamiento': PARAM['agrupamiento'], 'granularidad': GRANULARIDAD,
        'filtro_clientes': PARAM['filtro_clientes'], 'clientes_n': PARAM['clientes_n'],
        'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
        'meses_inferencia': sorted(MESES_INFER),
        'reentrenar_con_val_para_test': PARAM['reentrenar_con_val_para_test'],
        'metrica': 'wape_toneladas' + ('_por_producto' if PARAM['wape_por_producto'] else '_por_fila'),
        'hiperparametros': study.best_params, 'n_trials': len(study.trials),
        'wape_mejor_trial': study.best_value,
        'wape_val': wape_val, 'wape_val_por_mes': wape_val_mes,
        'wape_test': wape_test, 'wape_test_por_mes': wape_test_mes,
        'wape_naive_val': naive_val, 'wape_naive_test': naive_test,
        'n_filas_train': int(df_pd['periodo'].isin(MESES_TRAIN).sum()),
        'n_features': len(FEATURES), 'features': FEATURES, 'cat_features': CAT_FEATURES,
    }, _path_res)

L.etapa("modelo final", _path_res, _entrenar_y_evaluar, 'final' in PARAM['forzar'])
RES = json.load(open(_path_res, encoding='utf-8'))
print(f"\nWAPE val {RES['wape_val']:.5f}   test {RES['wape_test']:.5f}")

## 9 — Predicción de inferencia y armado de la entrega

Kaggle mide por `product_id`. Con granularidad producto-cliente hay que **sumar las
predicciones de todos los clientes** de cada producto, que es la misma agregación con
la que se midió el WAPE.

In [ ]:
_path_pred = DIR_OUT / 'pred_infer.parquet'
OBJ = PARAM['periodo_objetivo']

def _predecir():
    # El ensemble entrena un modelo por semilla y promedia: cancela parte de la
    # varianza del submuestreo de filas y columnas de LightGBM.
    mejores   = espacio_hiper(optuna.trial.FixedTrial(study.best_params))
    meses_fit = MESES_TRAIN + MESES_VAL + MESES_TEST     # aca se entrena con TODO
    preds = []
    for s in PARAM['semillas_ensemble']:
        p = dict(mejores); p.update({'seed': s, 'deterministic': True})
        print(f"   semilla {s} ...")
        m = entrenar(p, meses_fit)
        preds.append(m.predict(df_infer_pd[FEATURES]))
        del m; gc.collect()

    y = reconstruir_nivel(np.mean(preds, axis=0), df_infer_pd)
    y = np.maximum(y, PARAM['clip_min'])

    salida = df_infer_pd[[c for c in ('product_id', 'customer_id', 'periodo')
                          if c in df_infer_pd.columns]].copy()
    salida['tn_pred']          = y
    salida['periodo_objetivo'] = [int(((_idx(p) + H - 1) // 12) * 100
                                      + ((_idx(p) + H - 1) % 12) + 1)
                                  for p in salida['periodo']]
    L.escribir_parquet(pl.from_pandas(salida), _path_pred)

L.etapa("prediccion inferencia", _path_pred, _predecir, 'final' in PARAM['forzar'])

pred = pl.read_parquet(_path_pred).to_pandas()
print(f"\n{len(pred):,} predicciones   objetivos: {sorted(pred['periodo_objetivo'].unique())}")

_obj = pred[pred['periodo_objetivo'] == OBJ]
if _obj.empty:
    raise RuntimeError(f"No hay predicciones para {OBJ}. "
                       f"Disponibles: {sorted(pred['periodo_objetivo'].unique())}")

# Con granularidad producto-cliente hay que SUMAR sobre los clientes: Kaggle mide el
# total vendido de cada producto, la misma agregacion con la que se midio el WAPE.
_pred_prod = (_obj.groupby('product_id', as_index=False)['tn_pred'].sum()
                  .rename(columns={'tn_pred': 'tn'}))

# ── La entrega tiene que ser EXACTAMENTE la lista oficial ───────────────────
# Kaggle rechaza el archivo si sobra o falta una fila ("Submission must have N rows").
# Sobran cuando solo_target=False, porque el dataset trae los ~1233 productos y no
# solo los que hay que entregar. Faltan cuando un producto murio antes del periodo
# de inferencia: con denseLife deja de tener filas y no llega a la prediccion.
_oficial = pd.read_csv(L.DIR_RAW / "product_id_apredecir201912.txt", sep="\t")
_oficial = _oficial[['product_id']].drop_duplicates()
print(f"Lista oficial de Kaggle : {len(_oficial):,} productos")

submit = _oficial.merge(_pred_prod, on='product_id', how='left')
_faltan = submit['tn'].isna()
print(f"Con prediccion propia   : {(~_faltan).sum():,}")
print(f"Sin prediccion          : {int(_faltan.sum()):,}"
      + (f"  -> relleno '{PARAM['relleno_faltantes']}'" if _faltan.any() else ""))
_sobran = len(_pred_prod) - int((~_faltan).sum())
if _sobran > 0:
    print(f"Predichos fuera de la lista: {_sobran:,}  (se descartan, no van en la entrega)")

if _faltan.any():
    if PARAM['relleno_faltantes'] == 'naive' and 'tn0' in df_pd.columns:
        # Ultimo tn conocido de cada producto, sumado sobre clientes. Para un producto
        # que sigue vivo es mucho mejor estimador que 0.
        _ult = (df_pd.groupby(['product_id', 'periodo'], as_index=False)['tn0'].sum()
                     .sort_values('periodo')
                     .groupby('product_id', as_index=False)
                     .last()[['product_id', 'tn0']]
                     .rename(columns={'tn0': '_naive'}))
        submit = submit.merge(_ult, on='product_id', how='left')
        submit['tn'] = submit['tn'].fillna(submit['_naive'])
        submit = submit.drop(columns='_naive')
        print(f"   rellenados con su ultimo tn conocido: "
              f"{int(_faltan.sum() - submit['tn'].isna().sum()):,}")
    submit['tn'] = submit['tn'].fillna(0.0)

submit['tn'] = submit['tn'].clip(lower=PARAM['clip_min'])
submit = submit[['product_id', 'tn']].sort_values('product_id').reset_index(drop=True)

if len(submit) != len(_oficial):
    raise RuntimeError(f"La entrega tiene {len(submit)} filas y la lista oficial "
                       f"{len(_oficial)}. Kaggle la va a rechazar.")

print(f"\nEntrega: {len(submit):,} productos   tn total {submit['tn'].sum():,.0f}")
print(submit.head())

In [ ]:
_path_csv = DIR_OUT / f"submission_{OBJ}.csv"

def _csv():
    submit.to_csv(_path_csv, index=False)

L.etapa("submission", _path_csv, _csv, 'submission' in PARAM['forzar'])
shutil.copy(_path_csv, L.RUTA_EXP / "submission_ultima.csv")

# Chequeos antes de entregar: es la ultima oportunidad de detectar una entrega rota.
_p = pd.read_csv(_path_csv)
print(f"{_path_csv}\n{len(_p):,} filas   columnas: {list(_p.columns)}")
print(f"tn: min {_p['tn'].min():.3f}  media {_p['tn'].mean():.3f}  max {_p['tn'].max():.3f}")
if _p['tn'].isna().any():          print("[ALERTA] hay NaN en la entrega")
if (_p['tn'] < 0).any():           print("[ALERTA] hay toneladas negativas")
if _p['tn'].nunique() < 10:        print("[ALERTA] casi todas las predicciones son iguales")
if _p['product_id'].duplicated().any(): print("[ALERTA] product_id duplicados")
if len(_p) != len(_oficial):
    print(f"[ALERTA] {len(_p)} filas pero la lista oficial tiene {len(_oficial)}: "
          "Kaggle va a rechazar la entrega")
else:
    print(f"Conteo de filas OK: {len(_p):,} = lista oficial")

## 10 — Submit a Kaggle

In [ ]:
import subprocess

def kaggle_cli(args):
    """Corre la CLI de kaggle. Nunca lanza excepcion: el CSV ya esta generado y no
    vale la pena romper la corrida por un problema de red."""
    try:
        r = subprocess.run(['kaggle'] + args, capture_output=True, text=True, timeout=300)
        return r.returncode == 0, (r.stdout or '') + (r.stderr or '')
    except Exception as e:
        return False, str(e)

if not PARAM['submit']:
    print("submit=False: el CSV quedo generado pero NO se subio.")
    print(f"Para subirlo a mano:\n"
          f"   kaggle competitions submit -c {PARAM['kaggle_competition']} "
          f"-f {_path_csv} -m \"{EXPERIMENTO[:60]}\"")
elif not (Path.home() / '.kaggle' / 'kaggle.json').exists():
    print("Sin credenciales de Kaggle: no se sube. Subi kaggle.json al bucket.")
else:
    msg = PARAM['mensaje_submit'] or (
        f"{EXPERIMENTO[:70]} | wape_test={RES['wape_test']:.5f} | "
        f"{len(PARAM['semillas_ensemble'])} semillas")
    ok, salida = kaggle_cli(['competitions', 'submit',
                             '-c', PARAM['kaggle_competition'],
                             '-f', str(_path_csv), '-m', msg])
    print(salida if salida.strip() else "(sin respuesta)")
    print("Submit OK" if ok else "El submit fallo; el CSV esta generado y se puede subir a mano.")

## 11 — Leaderboard: comparar experimentos entre sí

In [ ]:
filas = []
for d in sorted(L.RUTA_EXP.iterdir()):
    f = d / 'resultado.json'
    if not (d.is_dir() and f.exists()):
        continue
    r = json.load(open(f, encoding='utf-8'))
    filas.append({
        'wape_test':  r.get('wape_test'),
        'wape_val':   r.get('wape_val'),
        'naive_test': r.get('wape_naive_test'),
        'granul':     r.get('granularidad'),
        'target':     r.get('target'),
        'clientes':   f"{r.get('filtro_clientes','?')}{r.get('clientes_n','')}",
        'trials':     r.get('n_trials'),
        'experimento': d.name[:52],
    })

if filas:
    lb = pd.DataFrame(filas).sort_values('wape_test', na_position='last')
    lb.to_csv(L.RUTA_EXP / 'leaderboard.csv', index=False)
    print(lb.to_string(index=False))
    print(f"\nGuardado: {L.RUTA_EXP / 'leaderboard.csv'}")
else:
    print("Todavia no hay experimentos con resultado.json")